In [ ]:
START = "Magetan" 
TARGET = "Surabaya"
METHOD = "astar" # greedy for GBFS, astar for A*


In [ ]:
import heapq
from collections import deque

graph = {
    "Magetan": [("Ngawi", 32), ("Madiun", 22), ("Ponorogo", 34)],
    "Ponorogo": [("Magetan", 34), ("Madiun", 29)],
    "Ngawi": [("Magetan", 32), ("Madiun", 30), ("Bojonegoro", 44)],
    "Madiun": [("Ngawi", 30), ("Magetan", 22), ("Ponorogo", 29), ("Nganjuk", 48)],
    "Bojonegoro": [("Ngawi", 44), ("Nganjuk", 33), ("Lamongan", 42), ("Jombang", 70)],
    "Nganjuk": [("Madiun", 48), ("Bojonegoro", 33), ("Jombang", 40)],
    "Lamongan": [("Bojonegoro", 42), ("Gresik", 14)],
    "Gresik": [("Lamongan", 14), ("Surabaya", 12)],
    "Jombang": [("Bojonegoro", 70), ("Nganjuk", 40), ("Surabaya", 72)],
    "Surabaya": [("Gresik", 12), ("Jombang", 72), ("Bangkalan", 44), ("Sidoarjo", 25)],
    "Sidoarjo": [("Surabaya", 25), ("Probolinggo", 78)],
    "Bangkalan": [("Surabaya", 44), ("Sampang", 52)],
    "Sampang": [("Bangkalan", 52), ("Pamekasan", 31)],
    "Pamekasan": [("Sampang", 31), ("Sumenep", 54)],
    "Sumenep": [("Pamekasan", 54)],
    "Probolinggo": [("Sidoarjo", 78), ("Situbondo", 99)],
    "Situbondo": [("Probolinggo", 99)],
}

def dijkstra(source):
    dist = {source: 0}
    pq = [(0, source)]
    while pq:
        d, node = heapq.heappop(pq)
        if d > dist[node]:
            continue
        for nxt, w in graph[node]:
            nd = d + w
            if nd < dist.get(nxt, float("inf")):
                dist[nxt] = nd
                heapq.heappush(pq, (nd, nxt))
    return dist

def pick_landmarks(k):
    first = next(iter(graph))
    landmarks = []
    tables = {}
    dist_to_set = dijkstra(first)
    for _ in range(k):
        farthest = max(graph, key=lambda c: dist_to_set[c])
        landmarks.append(farthest)
        tables[farthest] = dijkstra(farthest)
        dist_to_set = {c: min(dist_to_set[c], tables[farthest][c]) for c in graph}
    return landmarks, tables

landmarks, landmark_dist = pick_landmarks(3)

def build_heuristic(goal):
    return {
        city: max(abs(landmark_dist[l][city] - landmark_dist[l][goal]) for l in landmarks)
        for city in graph
    }


In [ ]:
def search(start, goal, method):
    h = build_heuristic(goal)
    counter = 0
    frontier = [(h[start], counter, 0, start)]
    parent = {start: None}
    best_g = {start: 0}
    closed = set()
    expanded = 0

    while frontier:
        _, _, g, node = heapq.heappop(frontier)
        if node in closed:
            continue
        closed.add(node)
        expanded += 1

        if node == goal:
            path = []
            while node is not None:
                path.append(node)
                node = parent[node]
            return path[::-1], g, expanded

        for nxt, w in graph[node]:
            ng = g + w
            if method == "greedy":
                if nxt in closed or nxt in parent:
                    continue
                parent[nxt] = node
                priority = h[nxt]
            else:
                if nxt in best_g and best_g[nxt] <= ng:
                    continue
                best_g[nxt] = ng
                parent[nxt] = node
                priority = ng + h[nxt]
            counter += 1
            heapq.heappush(frontier, (priority, counter, ng, nxt))

    return None, None, expanded


In [ ]:
lookup = {name.lower(): name for name in graph}
methods = {"greedy": "Greedy BFS", "astar": "A*"}

start = lookup.get(START.strip().lower())
goal = lookup.get(TARGET.strip().lower())
method = METHOD.strip().lower()

if start is None:
    print("Unknown START. Options:", ", ".join(graph))
elif goal is None:
    print("Unknown TARGET. Options:", ", ".join(graph))
elif method not in methods:
    print('METHOD must be "greedy" or "astar"')
else:
    path, cost, expanded = search(start, goal, method)
    print("Method        :", methods[method])
    print("From          :", start)
    print("To            :", goal)
    if path is None:
        print("No route found")
    else:
        print("Route         :", " -> ".join(path))
        print("Total cost    :", cost)
        print("Nodes expanded:", expanded)
